<a href="https://colab.research.google.com/github/Tmiller68/machine-learning-fundamentals/blob/main/Day_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import files
uploaded=files.upload()

Saving adult.data to adult.data


In [3]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
column_names = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education_num",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital_gain",
    "capital_loss",
    "hours_per_week",
    "native_country",
    "income"
]

df = pd.read_csv(
    "adult.data",
    names=column_names,
    header=None
)
df = df.replace(" ?", np.nan)
df = df.dropna()

In [4]:
X=df.drop("income", axis=1)
Y=df["income"].str.strip()
label_encoder=LabelEncoder()
Y=label_encoder.fit_transform(Y)

In [5]:
numeric_columns=X.select_dtypes(
    include=["int64", "float64"]
).columns
categorical_columns=X.select_dtypes(
    include=["object"]
).columns

In [8]:
print(numeric_columns)
print(categorical_columns)

Index(['age', 'fnlwgt', 'education_num', 'capital_gain', 'capital_loss',
       'hours_per_week'],
      dtype='object')
Index(['workclass', 'education', 'marital_status', 'occupation',
       'relationship', 'race', 'sex', 'native_country'],
      dtype='object')


In [10]:
preprocessor = ColumnTransformer([
    (
        "numeric",
        StandardScaler(),
        numeric_columns
    ),
    (
        "categorical",
        OneHotEncoder(handle_unknown="ignore"),
        categorical_columns
    )
])

In [11]:
adult_pipeline = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=2000,
            random_state=42
        )
    )
])

In [13]:
pipeline_scores = cross_val_score(
    adult_pipeline,
    X,
    Y,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

In [15]:
print("Pipeline fold scores:")
print(pipeline_scores)

print(
    "Pipeline mean accuracy:",
    pipeline_scores.mean()
)

print(
    "Pipeline standard deviation:",
    pipeline_scores.std()
)

Pipeline fold scores:
[0.84485331 0.84253274 0.85029841 0.85344828 0.84864058]
Pipeline mean accuracy: 0.8479546622664647
Pipeline standard deviation: 0.0038770893237290312


In [17]:
leaky_preprocessor = ColumnTransformer([
    (
        "numeric",
        StandardScaler(),
        numeric_columns
    ),
    (
        "categorical",
        OneHotEncoder(handle_unknown="ignore"),
        categorical_columns
    )
])
X=leaky_preprocessor.fit_transform(X)

In [19]:
leaky_model = LogisticRegression(
    max_iter=2000,
    random_state=42
)

leaky_scores = cross_val_score(
    leaky_model,
    X,
    Y,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

In [20]:
print("Leaked fold scores:")
print(leaky_scores)

print(
    "Leaked mean accuracy:",
    leaky_scores.mean()
)

print(
    "Leaked standard deviation:",
    leaky_scores.std()
)

Leaked fold scores:
[0.8445218  0.84253274 0.85013263 0.85311671 0.84864058]
Leaked mean accuracy: 0.8477888907648078
Leaked standard deviation: 0.003818512854146663


In [21]:
day4_results = pd.DataFrame({
    "Method": [
        "Leakage-Free Pipeline",
        "Preprocessing Before CV"
    ],
    "Mean CV Accuracy": [
        pipeline_scores.mean(),
        leaky_scores.mean()
    ],
    "Accuracy STD": [
        pipeline_scores.std(),
        leaky_scores.std()
    ]
})

day4_results

,Method,Mean CV Accuracy,Accuracy STD
0,Leakage-Free Pipeline,0.847955,0.003877
1,Preprocessing Before CV,0.847789,0.003819


In [22]:
day4_results["Mean CV Accuracy"] = (
    day4_results["Mean CV Accuracy"].round(4)
)

day4_results["Accuracy STD"] = (
    day4_results["Accuracy STD"].round(4)
)

day4_results

,Method,Mean CV Accuracy,Accuracy STD
0,Leakage-Free Pipeline,0.8480,0.0039
1,Preprocessing Before CV,0.8478,0.0038
